In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import sys
from importlib import reload

%matplotlib inline

import pyPamtra
help(pyPamtra)

## INSERIRE I DATI DEI DISDROMETRI (bins, edges)

In [ ]:
sys.platform

In [ ]:
def info_sistema():
    import platform

    # platform.release() restituisce la versione del kernel (es. '5.15.0-60-generic' vs '5.15.133.1-microsoft-standard-WSL2')
    kernel_version = platform.release().lower()
    
    if "microsoft" in kernel_version or "wsl" in kernel_version:
        base = "/mnt/c/Data/pamtra_db"
        print("Sei su WSL (Ubuntu)")
        return base
    else:
        base = "/home/roversi/work/pamtra_db"
        print("Sei su Linux nativo")
        return base

base = info_sistema()
base

In [ ]:
# if sys.platform == "darwin":
#     base = os.path.expanduser("~/Data/APP_MZS")

# if sys.platform == "win32":
#     base = "C:/Data/pamtra_db"
    
# else:  # Linux / HPC (tintin)
#     base = "/home/roversi/work/pamtra_db"

data = xr.load_dataset(f"{base}/data/haloac3_dataset_for_pamtra.nc")
data

In [ ]:
# psd_i = np.ma.masked_invalid(data.psd_ice.isel(time=0).values).filled(0)
# psd_l = np.ma.masked_invalid(data.psd_liquid.isel(time=0).values).filled(0)
psd_i = np.ma.masked_invalid(data.psd_ice.mean(dim='time', skipna=True).values).filled(0)
psd_l = np.ma.masked_invalid(data.psd_liquid.mean(dim='time', skipna=True).values).filled(0)
nBins = np.shape(psd_i)[0] #[1]

In [ ]:
# nBins = 32
nBins

In [ ]:
Dmean = data.bin_mid.values*1e-6 # bc values in microns
Dbound = np.append(data.bin_min.values[:,0], data.bin_max.values[-1, 0])
Dbound = Dbound*1e-6

In [ ]:
Dbound, Dmean

Next, we create an empty pyPamtra object

In [ ]:
pam = pyPamtra.pyPamtra()
pam.dimensions


Note that parameters not provided will be guessed, please make sure to look at the warning messages carefully. Now, the `pam.p` dictionary is created

In [ ]:
sorted(pam.p.keys())

In [ ]:
pam.df.addHydrometeor((
        "liquid",  # name 
        -99.,  # aspect ratio (NOT RELEVANT)
        1,  # liquid - ice flag
        -99.,  # density (NOT RELEVANT)
        -99.,  # mass size relation prefactor a (NOT RELEVANT)
        -99.,  # mass size relation exponent b (NOT RELEVANT)
        -99.,  # area size relation prefactor alpha (NOT RELEVANT)
        -99.,  # area size relation exponent beta (NOT RELEVANT)
        0,  # moment provided later (NOT RELEVANT)
        nBins,  # number of bins
        "fullBin",  # distribution name (NOT RELEVANT)
        -99.,  # distribution parameter 1 (NOT RELEVANT)
        -99.,  # distribution parameter 2 (NOT RELEVANT)
        -99.,  # distribution parameter 3 (NOT RELEVANT)
        -99.,  # distribution parameter 4 (NOT RELEVANT)
        -99.,  # minimum diameter (NOT RELEVANT)
        -99.,  # maximum diameter (NOT RELEVANT)
        'mie-sphere',  # scattering model
        'khvorostyanov01_drops',  # fall velocity relation
        0.0  # canting angle
    ))

In [ ]:
pam.df.addHydrometeor((
    "ice",  # name ## "liquid" -> "ice"
    -99.,  # aspect ratio (NOT RELEVANT)
    -1,  # liquid - ice flag
    -99.,  # density (NOT RELEVANT)
    -99.,  # mass size relation prefactor a (NOT RELEVANT)
    -99.,  # mass size relation exponent b (NOT RELEVANT)
    -99.,  # area size relation prefactor alpha (NOT RELEVANT)
    -99.,  # area size relation exponent beta (NOT RELEVANT)
    0,  # moment provided later (NOT RELEVANT)
    nBins,  # number of bins
    "fullBin",  # distribution name (NOT RELEVANT)
    -99.,  # distribution parameter 1 (NOT RELEVANT)
    -99.,  # distribution parameter 2 (NOT RELEVANT)
    -99.,  # distribution parameter 3 (NOT RELEVANT)
    -99.,  # distribution parameter 4 (NOT RELEVANT)
    -99.,  # minimum diameter (NOT RELEVANT)
    -99.,  # maximum diameter (NOT RELEVANT)
    'ss-rayleigh-gans',  # scattering model ## SSRGA
    'heymsfield10_particles',  # fall velocity relation ## more realistic for ice particles
    0.0  # canting angle
))

Next, we add an an atmospheric profile to PAMTRA. For this example, we use an US standard profile for simplicity. The `pyPamtra.importer.createUsStandardProfile` helper routine requires only height levels as an input. The height dimension of level properties is one longer than for layer properties such as hydrometeor properties. Therefore, we need to add one more height level to the ACME-V data set. Here, we simply add a height value to the vector. For comparison with e.g. a ground-based radar, the height levels should be derived from the height layers through interpolation. 

In [ ]:
pam = pyPamtra.importer.createUsStandardProfile(
    pam,
    hgt_lev=np.arange(1000,1300,200)
)

# TODO set correct altitude
# TODO set correct lat lon time

The temperature and pressure fields have been populated with US standard atmosphere values in K and Pa, respectively. Note that all input quantities in PAMTRA except frequency (GHz) are in SI units. Refer to `pam.units` for details.

In [ ]:
pam.p['temp_lev'], pam.p['press_lev']

To model turbulence properly, we need to define the horizontal wind speed and the eddy dissipation rate (see Maahn et al. 2015, https://doi.org/10.1175/JTECH-D-14-00112.1 for details). For simplicity, we choose fixed values for this example

In [ ]:
pam.p['wind_uv'][:] = 10
pam.p['turb_edr'][:] = 1e-4

# pam.p["airturb"][:] = 0.01

We set some non-default settings, see the documentation (https://pamtra.readthedocs.io/en/latest/settings.html) for details. Note the `pam.nmlSet["hydro_fullspec"] = True` which tells the Python Fortran kernel to use the provided distributions instead of the parameters of the hydrometeor description.

In [ ]:
pam.nmlSet["passive"] = False
pam.nmlSet["randomseed"] = 0 #100
pam.nmlSet["radar_mode"] = "spectrum"
pam.nmlSet["radar_aliasing_nyquist_interv"] = 2
pam.nmlSet["hydro_adaptive_grid"] = False
pam.nmlSet["conserve_mass_rescale_dsd"] = False
pam.nmlSet["radar_use_hildebrand"] = True
pam.nmlSet["radar_noise_distance_factor"] = -2
pam.nmlSet["hydro_fullspec"] = True

# pam.nmlSet["radar_aliasing_nyquist_interv"] = 12
# pam.nmlSet["save_psd"] = True
# pam.p["hydro_q"][:] = 1e-2


For debugging, verbosity of the Fortran and Python code can be increased. Note that due to technical limitations (https://github.com/ipython/ipykernel/issues/110) the output of the Fortran kernel does not show up in Jupyter. For debugging the Fortran kernel, you can start `iPython` in a terminal and run this script with `%run fullbin-acmev-example.ipynb` to see the debugging output.

In [ ]:
pam.set["verbose"] = 0
pam.set["pyVerbose"] = 0

Finally, we create the Python objects for the measured DSDs

In [ ]:
pam.df.addFullSpectra()

which creates the `pam.df.dataFullSpec` dictionary containing empty arrays which need to be populated 

In [ ]:
list(pam.df.dataFullSpec.keys())

We start with adding the in situ observations for `d_bound_ds` (size bin boundaries in m), `d_ds` (size bin center in m, used for scattering calculation), and `n_ds` (number concentration in 1/m$^3$). Note that the in situ data set contains a drop size distribution in 1/m$^4$ which is why we apply `np.diff(Dbound)`.

In [ ]:
pam.df.dataFullSpec["d_bound_ds"][:] = Dbound
pam.df.dataFullSpec["d_ds"][:] = Dmean
pam.df.dataFullSpec["n_ds"][0,0,:,0,:] = (psd_l) * np.diff(Dbound)  
pam.df.dataFullSpec['n_ds'][0,0,:,1,:] = (psd_i) * np.diff(Dbound)

Note that the dimension of these arrays is

In [ ]:
pam.df.dataFullSpec["n_ds"].shape

which is for x-dimension, y-dimension, height, hydrometeor type (in case there are more than one), and size bin. Therefore, `np.newaxis` needs to be used for the measured DSDs to allow broadcasting to the required shape. 

It is crucial to define also the other hydrometeor properties `rho_ds` (particle density in kg/m$^3$), `area_ds` (cross section area in m$^2$), `mass_ds` (particle mass in kg), and `as_ratio` (aspect ratio, oblate for values < 1). However, for liquid cloud and drizzle drops, the trivial relations for spheres can be used. This give the opportunity to use arbitrarily complex relations for ice and snow particles. Note that it is the sole responsibility of the user to make sure all relations are consistent with each other, unless in PAMTRA's normal mode, the full-bin interface does not do any consistency checks. 

#### liquid droplets:
assumed to be spherical; we use Mie scattering

In [ ]:
# liquid
pam.df.dataFullSpec["rho_ds"][0,0,:,0,:] = 1000.
pam.df.dataFullSpec["area_ds"][0,0,:,0,:] = (np.pi / 4. * pam.df.dataFullSpec["d_ds"][0,0,:,0,:]**2)
pam.df.dataFullSpec["mass_ds"][0,0,:,0,:] = (np.pi / 6. * pam.df.dataFullSpec["rho_ds"][0,0,:,0,:] * 
                                             pam.df.dataFullSpec["d_ds"][0,0,:,0,:]**3)
pam.df.dataFullSpec["as_ratio"][0,0,:,0,:] = 1.0

#### rimed ice particles:

Now we add scattering and physical ice partile properties for our rimed ice particle population. We can provide the normalized rime mass M as a scalar or vector (different value for each time step). 

Normalized rime mass retrival values from Maherndl et al. (2023b, submitted):

In [ ]:
# M = data.M.values

Alternatively: choose fixed M:      

In [ ]:
M = np.full(1, 0.01) # assume normalized rime mass
M

Mass (area) can be calculated from power law relations with prefactor a_m (a_a) and exponent b_m (b_a). ssrga_parameter(M) gives the SSRGA parameter kappa, beta, gamma, zeta_1 and alpha_eff as output. alpha_eff is equivalent to the aspect ratio here. 

In [ ]:
# rimed ice particles
a_m = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_mass_size(M, 'dendrite')[0], 1) # mass size prefactor
b_m = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_mass_size(M, 'dendrite')[1], 1) # mass size exponent
a_a = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_area_size(M, 'dendrite')[0], 1) # area size prefactor 
b_a = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_area_size(M, 'dendrite')[1], 1) # area size exponent 

pam.df.dataFullSpec["area_ds"][0,0,:,1,:] = (a_a * pam.df.dataFullSpec["d_ds"][0,0,:,1,:]**b_a)
pam.df.dataFullSpec["mass_ds"][0,0,:,1,:] = (a_m * pam.df.dataFullSpec["d_ds"][0,0,:,1,:]**b_m)

pam.df.dataFullSpec["as_ratio"][0,0,:,1,:]    = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[4], 1) # alpha_eff
pam.df.dataFullSpec["rg_kappa_ds"][0,0,:,1,:] = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[0], 1) 
pam.df.dataFullSpec["rg_beta_ds"][0,0,:,1,:]  = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[1], 1) 
pam.df.dataFullSpec["rg_gamma_ds"][0,0,:,1,:] = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[2], 1) 
pam.df.dataFullSpec["rg_zeta_ds"][0,0,:,1,:]  = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[3], 1)

Finally, we can run PAMTRA for the intended frequencies to estimate the radar observables. It is recommended to check whether `pam.fortError == 0` after running PAMTRA to catch errors in the Fortran part which are not displayed in Jupyter as discussed above. 

In [ ]:
frequencies = [24.15]  #[94]
pam.runPamtra(frequencies)
print((pam.fortError))

Now, we can analyze the results which are stored in the `pam.r` dictionary.

In [ ]:
list(pam.r.keys())

In [ ]:
# PAMTRA's out_debug_* arrays are fixed-size (300); only the first nBins are
# meaningful (see src/vars_output.f90:50). Slice to nBins to align with psd_i.
diameter  = pam.fortObject.vars_output.out_debug_diameter[:nBins]   # SI:     [m]
back_spec = pam.fortObject.vars_output.out_debug_back_of_d[:nBins]  # non-SI: [mm6/m³/m]

diameter = np.where(diameter == 0, np.nan, diameter)

In [ ]:
plt.plot(diameter)

In [ ]:
plt.figure()
plt.plot(diameter, back_spec, 'o')
plt.xlabel("D [m]")
plt.ylabel("Backscattering [mm6 / m3 / m]")
plt.title(f"Backscattering per diameter | rimed fraction M = 0.01")
plt.yscale("log")

In [ ]:
# ---------------------------------------------------------------------------
# CORRECTION (2026-05-26): back_spec is dZ/dD in mm⁶/m³/m, NOT per-particle σ_b.
# From src/radar_spectrum.f90 (line 41 & 232):
#   input back_spec [m²/m⁴] = σ_b(D) · N(D)              (includes PSD)
#   out_debug_back_of_d = back_spec · λ⁴ / (K²·π⁵) · 1e18   [mm⁶/m³/m]   (= dZ/dD)
#
# The previous `cumulative_trapezoid` produced Z(D) (cumulative reflectivity),
# NOT a backscatter cross section. To recover the PER-PARTICLE σ_b(D) [m²]
# we divide by N(D) [1/m⁴] and undo the K²/λ⁴ normalisation. No cumtrapz.
# ---------------------------------------------------------------------------

K2 = 0.92
freq = 24.15e9                              # Hz
lm = 299792458. / freq                      # m

# N(D) on the same grid as `diameter`; psd_i is the PSD in 1/m⁴ (= 1/m³/m)
N_D = psd_i

with np.errstate(divide='ignore', invalid='ignore'):
    sigma_b_pp = back_spec * (K2 * np.pi**5) / (1e18 * lm**4 * N_D)   # [m²]
sigma_b_pp = np.where(N_D > 0, sigma_b_pp, np.nan)

sigma_b_pp

In [ ]:
plt.figure()
plt.plot(diameter * 1e3, sigma_b_pp, 'o')
plt.xlabel("D [mm]")
plt.ylabel(r"$\sigma_b$ per particle [m$^2$]")
plt.title("Per-particle backscatter cross section | M = 0.01, 24 GHz")
plt.xscale("log"); plt.yscale("log")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
M

In [ ]:
K2 = 0.92

freq = 24.15e9 # Hz
#freq = 24.29e9 # Hz (post 2011) TODO re-run with this! (M&K also!)
lm = 299792458. / freq

# sigma_ref = (1e18/ (K2*np.pi**5) ) * sigma_D * (lm)**4
(K2 * np.pi**5) / (1e18 * (lm)**4)

In [ ]:
pyPamtra.descriptorFile.ssrga_parameter(M)[4]

In [ ]:
ea = 1/pyPamtra.descriptorFile.ssrga_parameter(M)[4]

In [ ]:
Dmax_mm = diameter * 1e3 * ea

In [ ]:
plt.figure()
plt.plot(Dmax_mm, sigma_b_pp, 'o')
plt.xlabel(r"$D_{max}$ [mm]")
plt.ylabel(r"$\sigma_b$ per particle [m$^2$]")
plt.title("Per-particle backscatter cross section vs $D_{max}$ | M = 0.01, 24 GHz")
plt.xscale("log"); plt.yscale("log")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
diameter.shape

In [ ]:
1e0

In [ ]:
np.pi

### Iterate for M
Compute and plot backscattering cross section (sigma_b) at 24 GHz
for an array of particle diameters (mm) and rimed fractions M
using pyPamtra SSRGA standard ensemble.

In [ ]:
# ---------------------- USER INPUT ----------------------
diameters_mm = np.linspace(0.05, 25, round(25/0.05))   # diameters in mm
M_list = (10**np.linspace(-2, 0, 11)).round(2)         # rimed fraction 0..1
# --------------------------------------------------------

# Prepare 2D array to store sigma_b and Dmax_mm
sigma_b_array = np.zeros((len(diameters_mm), len(M_list)))
Dmax_mm_array = np.zeros((len(diameters_mm), len(M_list)))


In [ ]:
# Iterate over M — extract PER-PARTICLE σ_b(D) for each rime mass fraction.
# Same correction as the single-M cell above: no cumtrapz; divide by N(D).
for i, Mint in enumerate(M_list):
    M = np.full(1, Mint)

    pam.df.addFullSpectra()

    ## Configure SSRGA
    pam.df.dataFullSpec["d_bound_ds"][:] = Dbound
    pam.df.dataFullSpec["d_ds"][:] = Dmean
    pam.df.dataFullSpec["n_ds"][0,0,:,0,:] = (psd_l) * np.diff(Dbound)
    pam.df.dataFullSpec['n_ds'][0,0,:,1,:] = (psd_i) * np.diff(Dbound)

    # liquid droplets
    pam.df.dataFullSpec["rho_ds"][0,0,:,0,:] = 1000.
    pam.df.dataFullSpec["area_ds"][0,0,:,0,:] = (np.pi / 4. * pam.df.dataFullSpec["d_ds"][0,0,:,0,:]**2)
    pam.df.dataFullSpec["mass_ds"][0,0,:,0,:] = (np.pi / 6. * pam.df.dataFullSpec["rho_ds"][0,0,:,0,:] *
                                                 pam.df.dataFullSpec["d_ds"][0,0,:,0,:]**3)
    pam.df.dataFullSpec["as_ratio"][0,0,:,0,:] = 1.0

    # rimed ice particles
    a_m = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_mass_size(M, 'dendrite')[0], 1)
    b_m = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_mass_size(M, 'dendrite')[1], 1)
    a_a = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_area_size(M, 'dendrite')[0], 1)
    b_a = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_area_size(M, 'dendrite')[1], 1)

    pam.df.dataFullSpec["area_ds"][0,0,:,1,:] = (a_a * pam.df.dataFullSpec["d_ds"][0,0,:,1,:]**b_a)
    pam.df.dataFullSpec["mass_ds"][0,0,:,1,:] = (a_m * pam.df.dataFullSpec["d_ds"][0,0,:,1,:]**b_m)

    pam.df.dataFullSpec["as_ratio"][0,0,:,1,:]    = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[4], 1)
    pam.df.dataFullSpec["rg_kappa_ds"][0,0,:,1,:] = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[0], 1)
    pam.df.dataFullSpec["rg_beta_ds"][0,0,:,1,:]  = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[1], 1)
    pam.df.dataFullSpec["rg_gamma_ds"][0,0,:,1,:] = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[2], 1)
    pam.df.dataFullSpec["rg_zeta_ds"][0,0,:,1,:]  = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[3], 1)

    pam.runPamtra(frequencies)
    if pam.fortError != 0:
        warnings.warn('Fortran errors!')

    # Slice fixed-size (300) PAMTRA debug outputs to actual nBins
    diameter  = pam.fortObject.vars_output.out_debug_diameter[:nBins]   # [m]
    back_spec = pam.fortObject.vars_output.out_debug_back_of_d[:nBins]  # [mm⁶/m³/m]
    diameter  = np.where(diameter == 0, np.nan, diameter)

    # ---- PER-PARTICLE σ_b extraction (no cumtrapz!) -----------------------
    # σ_b(D) [m²] = back_spec · (K² π⁵) / (1e18 λ⁴ N(D))
    with np.errstate(divide='ignore', invalid='ignore'):
        sigma_b_pp = back_spec * (K2 * np.pi**5) / (1e18 * lm**4 * psd_i)
    sigma_b_pp = np.where(psd_i > 0, sigma_b_pp, np.nan)

    # Convert D to D_max using α_eff for the current M (PAMTRA stores D_eff)
    ea = 1.0 / pyPamtra.descriptorFile.ssrga_parameter(M)[4]
    Dmax_mm_variable = diameter * 1e3 * ea                                 # [mm]

    # Interpolate onto the requested uniform D grid (NaNs preserved).
    # np.interp requires monotonic non-NaN x; mask first.
    ok = np.isfinite(Dmax_mm_variable) & np.isfinite(sigma_b_pp)
    if ok.sum() >= 2:
        order = np.argsort(Dmax_mm_variable[ok])
        sigma_b_uniform = np.interp(
            diameters_mm,
            Dmax_mm_variable[ok][order],
            sigma_b_pp[ok][order],
            left=np.nan, right=np.nan,
        )
    else:
        sigma_b_uniform = np.full_like(diameters_mm, np.nan)

    sigma_b_array[:, i] = sigma_b_uniform
    Dmax_mm_array[:, i] = diameters_mm

In [ ]:
diameters_mm

In [ ]:
Dmax_mm_variable

In [ ]:
# sigma_b_array = np.where(sigma_b_array == 0, np.nan, sigma_b_array)
sigma_b_array

In [ ]:
# Save CSV — note: now stores PER-PARTICLE σ_b(D, M) in m², not η.
df = pd.DataFrame(sigma_b_array, index=diameters_mm, columns=M_list)
df.index.name = 'Dmax_mm'
df.columns.name = 'M'
df.to_csv("sigma_b_perparticle_24ghz.csv")
print("CSV saved: sigma_b_perparticle_24ghz.csv  (units: m²)")

In [ ]:
cmap = plt.get_cmap('plasma')
colors = cmap(np.linspace(0, 1, len(M_list)))

plt.figure(figsize=(10, 6))
for i, M in enumerate(M_list):
    plt.plot(diameters_mm, sigma_b_array[:, i], color=colors[i], label=f'M={M:.2f}')

plt.xlabel(r"$D_{max}$ [mm]")
plt.ylabel(r"$\sigma_b$ per particle [m$^2$]")
plt.title("PAMTRA SSRGA dendrite — per-particle backscatter at 24 GHz")
#plt.xscale("log");
plt.yscale("log")
plt.grid(axis='both', which='both', alpha=0.3)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig("sigma_b_perparticle_24ghz.png", dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
M_list